# Prepare reliability data

## Setup

In [ ]:
import pandas as pd
import csv
# get working directory
import os
os.getcwd()
# set working directory
os.chdir('analyses/annotation/reliability')

In [ ]:
# Import spss file
df = pd.read_spss('nos_coded_final.sav')
print(df.shape)

In [ ]:
# Import spss file
df_extra = pd.read_spss('nos_coded_second_coder_extra.sav')
print(df_extra.shape)

In [ ]:
# concatenate the two dataframes
df = pd.concat([df, df_extra])
print(df.shape)

In [ ]:
# see where article_id is duplicated
df['M2'] = df['M2'].astype(int)

In [ ]:
# rename columns M1, M2, M3, M4 
df = df.rename(columns={'M1': 'coder', 
                        'M2': 'article_id', 
                        'M3': 'title', 
                        'M4': 'owner'})

In [ ]:
# remove if Finished is False
df = df[df.Finished == 'True']

In [ ]:
duplicates = df[df.duplicated(['article_id', 'coder'], keep=False)]

In [ ]:
df[df['article_id'].isin(duplicates.article_id)][['article_id', 'coder', 'RecordedDate']]

In [ ]:
# keep the last duplicate
df = df.drop_duplicates(subset=['article_id', 'coder'], keep='last')

In [ ]:
df[df['article_id'].isin(duplicates.article_id)][['article_id', 'coder', 'RecordedDate']]

In [ ]:
# find the columns where all values are missing
for i in df.columns[df.isnull().all()]:
    print(i)

In [ ]:
# get the columns after 17th column
df = df.iloc[:, 17:]

In [ ]:
# count the unique coders per article_id
article_ids_second_coder = df[df['coder'] == 'second_coder']['article_id'].unique()

In [ ]:
# reliability dataframe
reliability_df = df[df['article_id'].isin(article_ids_second_coder)]
print(len(reliability_df))

In [ ]:
# rename columns M1, M2, M3, M4 
reliability_df = reliability_df.rename(columns={'V0_': 'about_covid', 
                        'V1._1': 'topic_a', 
                        'V1._2': 'topic_b', 
                        'V1._21': 'topic_c',    
                        'V1._3': 'topic_d',
                        'V1._4': 'topic_e',
                        'V1._5': 'topic_f',
                        'V1._6': 'topic_g',
                        'V1._7': 'topic_h',
                        'V1._8': 'topic_i',
                        'V1._9': 'topic_j',
                        'V1._10': 'topic_k',
                        'V1._11': 'topic_l',
                        'V1._12': 'topic_m',
                        'V1._13': 'topic_n',
                        'V1._15': 'topic_o',
                        'V1._15_TEXT': 'topic_o_text', 
                        'V1.2._2': 'other_country_binary',
                        'V2': 'actors_present'
                        })

In [ ]:
# Filter columns that start with 'V1.3._'
country_columns = reliability_df.filter(like='V1.3._')

# Concatenate non-NaN values across columns into a new column
reliability_df['country_name'] = country_columns.apply(lambda row: ', '.join(row.dropna().astype(str)), axis=1)

# Drop the original country-coded columns
reliability_df = reliability_df.drop(columns=country_columns.columns)

In [ ]:
for i in reliability_df.columns:
    print(i)

In [ ]:
# save reliability_df in utf-8 encoding
reliability_df.to_csv('reliability_df_final_extra.csv', index=False, sep=';', quoting=csv.QUOTE_NONNUMERIC, encoding='utf-8')

# 1. Create the topic DF for reliability

In [ ]:
topic_df = reliability_df.loc[:, ['coder',
'article_id',
'title',
'owner',
'about_covid',
'topic_a',
'topic_b',
'topic_c',
'topic_d',
'topic_e',
'topic_f',
'topic_g',
'topic_h',
'topic_i',
'topic_j',
'topic_k',
'topic_l',
'topic_m',
'topic_n',
'topic_o',
'topic_o_text',
'other_country_binary',
'country_name',
'actors_present']]

In [ ]:
# if about_covid is Ja then 1 else 0
topic_df['about_covid'] = topic_df['about_covid'].apply(lambda x: 1 if x == 'Ja' else 0)
topic_df['actors_present'] = topic_df['actors_present'].apply(lambda x: 1 if x == 'Ja' else 0)

In [ ]:
# make missing 0
topic_df.actors_present.fillna(0, inplace=True)

In [ ]:
for i in topic_df.columns:
    if i.startswith('topic_'):

In [ ]:
# drop topic_o and topic_o_text
topic_df = topic_df.drop(columns=['topic_o', 'topic_o_text'])

In [ ]:
def change_to_binary(x):
    # if x is NaN then 0 else 1
    if pd.isna(x):
        return 0
    else:
        return 1

# Assuming df is your DataFrame
topic_columns = topic_df.filter(like='topic_')

reliability_df[topic_columns.columns] = topic_columns.applymap(change_to_binary)
topic_df[topic_columns.columns] = topic_columns.applymap(change_to_binary)

In [ ]:
for i in topic_df.columns:
    if i.startswith('topic_'):

In [ ]:
# make all topic columns integer
topic_df[topic_columns.columns] = topic_df[topic_columns.columns].applymap(int)

In [ ]:
topic_df['country_name'] = topic_df['country_name'].replace({'United States': 'US', 
                                                             'United Kingdom of Great Britain and Northern Ireland': 'UK', 
                                                             'United Arab Emirates': 'UAE'}, regex=True)

In [ ]:
# change to binary
topic_df['other_country_binary'] = topic_df['other_country_binary'].map(change_to_binary)

In [ ]:
pd.crosstab(topic_df['about_covid'], topic_df['actors_present'], dropna=False)

In [ ]:
# write topic_df to csv
topic_df.to_csv('reliability_topics_final_extra.csv', sep = ';', encoding = 'utf-8', quoting=csv.QUOTE_NONNUMERIC, index=False)

In [ ]:
print(len(topic_df))

# Create the Actors Dataframe

In [ ]:
reliability_df['about_covid'] = reliability_df['about_covid'].map({'Ja': 1, 'Nee': 0})
reliability_df['actors_present'] = reliability_df['actors_present'].map({'Ja': 1, 'Nee': 0})

In [ ]:
# article ids where about_covid is 1
article_ids_covid = reliability_df[reliability_df['about_covid'] == 1]['article_id'].unique()

In [ ]:
reliability_df['actors_present'] = reliability_df['actors_present'].fillna(0)

In [ ]:
# drop all topic columns
actors_df = reliability_df.drop(columns=topic_columns.columns)

In [ ]:
# drop if article not about covid
actors_df = actors_df[actors_df['article_id'].isin(article_ids_covid)]

In [ ]:
actors_df[actors_df['actors_present'].isnull()]

In [ ]:
# drop if actors_present is NaN
actors_df = actors_df.dropna(subset=['actors_present'])

In [ ]:
# drop columns topic_o_text other_country_binary, country_name
actors_df = actors_df.drop(columns=['topic_o_text', 'other_country_binary', 'country_name'])

In [ ]:
print(actors_df.isnull().sum())

In [ ]:
# see where actors_present is 0

actors_df[actors_df['actors_present'] == 0]

In [ ]:
actor_numbers = [str(i) for i in range(1, 26)]  # Assuming actors are labeled from A1 to A25

# Create a dictionary to store DataFrames for each actor
actor_dataframes = {}

# Iterate through actor numbers
for actor_number in actor_numbers:
    # Select columns related to the current actor
    actor_columns = [col for col in actors_df.columns if col.startswith(f'A{actor_number}_')]
    actor_columns.extend(['coder', 'article_id', 'title', 'about_covid', 'actors_present'])
    
    # Create a new DataFrame for the current actor
    actor_df = actors_df[actor_columns].copy()
    
    # Rename columns to remove the actor_number prefix
    actor_df.columns = [col.replace(f'A{actor_number}_', '') for col in actor_df.columns]
    
    # Store the DataFrame in the dictionary with the actor_number as the key
    actor_dataframes[actor_number] = actor_df

# Display one of the DataFrames, e.g., for actor A1
print(actor_dataframes['1'])

In [ ]:
all_actor_data = pd.concat(actor_dataframes.values(), ignore_index=True)

In [ ]:
all_actor_data[all_actor_data['V4.'].isnull()]

In [ ]:
# remove if V4 is null and actors_present is 1
all_actor_data = all_actor_data[~((all_actor_data['V4.'].isnull()) & (all_actor_data['actors_present'] == 1))]

In [ ]:
for i in all_actor_data.columns:
    print(i)

In [ ]:
# change column names
all_actor_data = all_actor_data.rename(columns={'V3': 'actor_name',
                                                'V4.': 'actor_type',
                                                'V5.': 'actor_function',
                                                'V5._20_TEXT': 'actor_function_text',
                                                'V5.1.': 'actor_pp',
                                                'V6._1': 'directly_quoted',
                                                'V6._2': 'indirectly_quoted',
                                                'V6._3': 'quoted_by_name',
                                                'V7.' : 'nr_words',
                                                'V8.': 'talks_covid_measures',
                                                'V9._1': 'measure_1',
                                                'V9.1._1': 'measure_1_positive',
                                                'V9.1._2': 'measure_1_negative',
                                                'V9.1._3': 'measure_1_neutral',
                                                'V9._2': 'measure_2',
                                                'V9.2._1': 'measure_2_positive',
                                                'V9.2._2': 'measure_2_negative',
                                                'V9.2._3': 'measure_2_neutral',
                                                'V9._3': 'measure_3',
                                                'V9.3._1': 'measure_3_positive',
                                                'V9.3._2': 'measure_3_negative',
                                                'V9.3._3': 'measure_3_neutral',
                                                'V9._4': 'measure_4',
                                                'V9.4._1': 'measure_4_positive',
                                                'V9.4._2': 'measure_4_negative',
                                                'V9.4._3': 'measure_4_neutral',
                                                'V9._5': 'measure_5',
                                                'V9.5._1': 'measure_5_positive',
                                                'V9.5._2': 'measure_5_negative',
                                                'V9.5._3': 'measure_5_neutral',
                                                'V9._6': 'measure_6',
                                                'V9.6._1': 'measure_6_positive',
                                                'V9.6._2': 'measure_6_negative',
                                                'V9.6._3': 'measure_6_neutral',
                                                'V9._7': 'measure_7',
                                                'V9.7._1': 'measure_7_positive',
                                                'V9.7._2': 'measure_7_negative',
                                                'V9.7._3': 'measure_7_neutral',
                                                'V9._8': 'measure_8',
                                                'V9.8._1': 'measure_8_positive',
                                                'V9.8._2': 'measure_8_negative',
                                                'V9.8._3': 'measure_8_neutral',
                                                'V9._9': 'measure_9',
                                                'V9.9._1': 'measure_9_positive',
                                                'V9.9._2': 'measure_9_negative',
                                                'V9.9._3': 'measure_9_neutral',
                                                'V9._10': 'measure_10',
                                                'V9.10._1': 'measure_10_positive',
                                                'V9.10._2': 'measure_10_negative',
                                                'V9.10._3': 'measure_10_neutral',
                                                'V9._11': 'measure_11',
                                                'V9.11._1': 'measure_11_positive',
                                                'V9.11._2': 'measure_11_negative',
                                                'V9.11._3': 'measure_11_neutral',
                                                'V9._12': 'measure_12',
                                                'V9.12._1': 'measure_12_positive',
                                                'V9.12._2': 'measure_12_negative',
                                                'V9.12._3': 'measure_12_neutral',
                                                'V9._13': 'measure_13',
                                                'V9.13._1': 'measure_13_positive',
                                                'V9.13._2': 'measure_13_negative',
                                                'V9.13._3': 'measure_13_neutral',
                                                'V9._14': 'measure_14',
                                                'V9.14._1': 'measure_14_positive',
                                                'V9.14._2': 'measure_14_negative',
                                                'V9.14._3': 'measure_14_neutral',
                                                'V9._15': 'measure_15',
                                                'V9.15._1': 'measure_15_positive',
                                                'V9.15._2': 'measure_15_negative',
                                                'V9.15._3': 'measure_15_neutral',
                                                'V9._16': 'measure_16',
                                                'V9.16._1': 'measure_16_positive',
                                                'V9.16._2': 'measure_16_negative',
                                                'V9.16._3': 'measure_16_neutral',
                                                'V9._18': 'measure_17',
                                                'V9.18._1': 'measure_17_positive',
                                                'V9.18._2': 'measure_17_negative',
                                                'V9.18._3': 'measure_17_neutral',
                                                'V9._17': 'measure_other',
                                                'V9.17._1': 'measure_other_positive',
                                                'V9.17._2': 'measure_other_negative',
                                                'V9.17._3': 'measure_other_neutral',
                                                'V9._17_TEXT': 'measure_other_text',
                                                'V10.': 'add_more_actor'
                                                })

In [ ]:
for i in all_actor_data.columns:
    if i.startswith('measure'):

In [ ]:
# get measure columns
measure_columns = all_actor_data.filter(like='measure_')
# drop measure_other_text column
measure_columns = measure_columns.drop(columns=['measure_other_text'])

# change measures to binary
all_actor_data[measure_columns.columns] = measure_columns.applymap(change_to_binary)

In [ ]:
for i in all_actor_data.columns:
    if i.startswith('measure'):

In [ ]:
for i in all_actor_data.columns:

In [ ]:
# apply change_to_binary function to directly_quoted, indirectly_quoted
all_actor_data['directly_quoted'] = all_actor_data['directly_quoted'].map(change_to_binary)
all_actor_data['indirectly_quoted'] = all_actor_data['indirectly_quoted'].map(change_to_binary)

# change talks_covid_measures to binary if Ja then 1 else 0
all_actor_data['talks_covid_measures'] = all_actor_data['talks_covid_measures'].map({'Ja': 1, 'Nee': 0})

In [ ]:
for i in all_actor_data.columns:

In [ ]:
for i in all_actor_data.columns:    
    print(i)

In [ ]:
column_order = ['coder', 'article_id','title','about_covid','actors_present','actor_name','actor_type','actor_function','actor_function_text','actor_pp','directly_quoted','indirectly_quoted','nr_words',
                'talks_covid_measures', 'measure_1', 'measure_1_positive', 'measure_1_negative', 'measure_1_neutral', 'measure_2', 'measure_2_positive', 'measure_2_negative', 'measure_2_neutral', 
                'measure_3', 'measure_3_positive', 'measure_3_negative', 'measure_3_neutral', 
                'measure_4', 'measure_4_positive', 'measure_4_negative', 'measure_4_neutral', 
                'measure_5', 'measure_5_positive', 'measure_5_negative', 'measure_5_neutral', 
                'measure_6', 'measure_6_positive', 'measure_6_negative', 'measure_6_neutral', 
                'measure_7', 'measure_7_positive', 'measure_7_negative', 'measure_7_neutral', 
                'measure_8', 'measure_8_positive', 'measure_8_negative', 'measure_8_neutral', 
                'measure_9', 'measure_9_positive', 'measure_9_negative', 'measure_9_neutral', 
                'measure_10', 'measure_10_positive', 'measure_10_negative', 'measure_10_neutral', 
                'measure_11', 'measure_11_positive', 'measure_11_negative', 'measure_11_neutral', 
                'measure_12', 'measure_12_positive', 'measure_12_negative', 'measure_12_neutral', 
                'measure_13', 'measure_13_positive', 'measure_13_negative', 'measure_13_neutral', 
                'measure_14', 'measure_14_positive', 'measure_14_negative', 'measure_14_neutral', 
                'measure_15', 'measure_15_positive', 'measure_15_negative', 'measure_15_neutral', 
                'measure_16', 'measure_16_positive', 'measure_16_negative', 'measure_16_neutral', 
                'measure_17', 'measure_17_positive', 'measure_17_negative', 'measure_17_neutral', 
                'measure_other', 'measure_other_positive', 'measure_other_negative', 'measure_other_neutral', 'add_more_actor']

In [ ]:
all_actor_data = all_actor_data[column_order]

In [ ]:
# write actors data to csv
all_actor_data.to_csv('reliability_actors_final_extra.csv', sep = ';', encoding = 'utf-8', quoting=csv.QUOTE_NONNUMERIC, index=False)

# Merge Actor and Topic DF's

In [ ]:
# add topic data to all_actor_data
all_actor_data_merged = pd.merge(all_actor_data, topic_df, on=['coder', 'article_id', 'title', 'about_covid', 'actors_present'], how='left')

In [ ]:
# crosstab actors_present with actor_name
pd.crosstab(all_actor_data_merged['actors_present'], all_actor_data_merged['actor_name'])

In [ ]:
# crosstab topic variables with actor_function
pd.crosstab(all_actor_data_merged['actor_function'], all_actor_data_merged['topic_a'])

In [ ]:
pd.crosstab(all_actor_data_merged['actor_function'], all_actor_data_merged['topic_b'])

In [ ]:
pd.crosstab(all_actor_data_merged['actor_function'], all_actor_data_merged['topic_c'])

In [ ]:
# write the merged data to csv
all_actor_data_merged.to_csv('reliability_merged_final_extra.csv', sep = ';', encoding = 'utf-8', quoting=csv.QUOTE_NONNUMERIC, index=False)